In [1]:
import pandas as pd
import re
import numpy as np


input_file = "core_articles.csv"
output_file = "core_articles_fixed.csv"


def clean_core_text(text):
    """
    Funkcja czyszcząca tekst:
    1. Usuwa entery i wielokrotne spacje (spłaszcza rozjechane wiersze).
    2. Usuwa tagi HTML/XML.
    3. Usuwa "urwane" tagi specyficzne dla CORE (np. jats).
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Spłaszczanie wierszy (zamiana \n, \r, \t na spację)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Usuwanie standardowych tagów (to co jest pomiędzy < a >)
    # Usuwa <i>, </i>, <p>, <jats:p> jeśli są domknięte
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. Fix specjalny - usuwanie niedomkniętych tagów jats (np. ...</jats:p)
    ### szuka ciągu zaczynającego się od <jats lub </jats i usuwa go nawet jeśli nie ma końcowego zamknięcia >
    ### nie niszcząc przylepionych cudzysłowów, przecinków
    text = re.sub(r'</?jats:\w+>?', '', text)
    
    return text

print(f"Wczytywanie pliku {input_file}...")
try:
    ### użycie on_bad_lines='skip' na wypadek tragicznie uszkodzonych linii
    #### pandas zazwyczaj radzi sobie z multiline jeśli cudzysłowy są ok
    df = pd.read_csv(input_file, on_bad_lines='skip')
    print(f"Wczytano {len(df):,} wierszy.")
except Exception as e:
    print(f"Błąd krytyczny przy wczytywaniu: {e}")
    df = pd.DataFrame()

if not df.empty:
    # 1. Czyszczenie tekstów (Title, Abstract)
    print("Rozpoczynam czyszczenie tagów i spłaszczanie wierszy...")
    cols_to_clean = ['title', 'abstract', 'keywords']
    
    for col in cols_to_clean:
        if col in df.columns:
            ## konwersja na string i aplikacja funkcji czyszczącej
            df[col] = df[col].astype(str).apply(clean_core_text)
            ## przywracanie pustych stringów jako NaN
            df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)
            df[col] = df[col].replace('nan', np.nan) ### usuwanie stringa "nan"

    # 2. Naprawa roku (np. 2009.0 -> 2009)
    if 'year' in df.columns:
        print("Formatowanie kolumny 'year'...")
        ## konwersja na numeryczny, błędy na NaN
        df['year'] = pd.to_numeric(df['year'], errors='coerce')
        ## zapis jako Int64 (typ, który obsługuje NaN, w przeciwieństwie do zwykłego int)
        df['year'] = df['year'].astype('Int64')

    # 3. Deduplikacja
    # sprawdzenie czy kolumna 'creator' istnieje (w fragmencie jej nie było w nagłówku, ale może być w danych)
    dedup_subset = ['title', 'abstract']
    if 'creator' in df.columns:
        dedup_subset.append('creator')
        df['creator'] = df['creator'].astype(str).apply(clean_core_text).replace('nan', np.nan)
    
    print(f"Deduplikacja w oparciu o kolumny: {dedup_subset}...")
    initial_count = len(df)
    
    df = df.drop_duplicates(subset=dedup_subset, keep='first')
    final_count = len(df)
    
    print(f"Usunięto {initial_count - final_count:,} duplikatów.")
    print(f"Pozostało {final_count:,} unikalnych rekordów.")

    # 4. Usuwanie śmieci (wiersze bez tytułu)
    print("Usuwanie wierszy bez tytułu...")
    before_drop = len(df)
    df = df.dropna(subset=['title'])
    after_drop = len(df)
    if before_drop > after_drop:
        print(f"Usunięto {before_drop - after_drop:,} wierszy, które nie miały tytułu.")

    # 5. Poprawka pustych keywords (uzupełnianie z kolumny query)
    print("Uzupełnianie brakujących keywords wartościami z query...")
    if 'query' in df.columns and 'keywords' in df.columns:
        # zliczanie ile brakowało przed operacją (dla statystyki)
        missing_count = df['keywords'].isna().sum()
        
        # wpisuje query tam, gdzie keywords jest puste
        df['keywords'] = df['keywords'].fillna(df['query'])
        
        print(f"Uzupełniono {missing_count:,} pustych pól w 'keywords'.")
    
    # 6. Zapis
    print(f"Zapisywanie do {output_file}...")
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    # podgląd
    print("\nPrzykładowe dane po naprawie")
    print(df[['title', 'year']].head(3))
    if 'abstract' in df.columns:
        print("\nFragment abstraktu (bez tagów):")
        # pierwszy niepusty abstrakt
        sample_abstract = df['abstract'].dropna().iloc[0] if not df['abstract'].dropna().empty else "Brak abstraktów"
        print(sample_abstract[:200] + "...")


Wczytywanie pliku core_articles.csv...
Wczytano 3,906 wierszy.
Rozpoczynam czyszczenie tagów i spłaszczanie wierszy...
Formatowanie kolumny 'year'...
Deduplikacja w oparciu o kolumny: ['title', 'abstract']...
Usunięto 331 duplikatów.
Pozostało 3,575 unikalnych rekordów.
Usuwanie wierszy bez tytułu...
Uzupełnianie brakujących keywords wartościami z query...
Uzupełniono 3,575 pustych pól w 'keywords'.
Zapisywanie do core_articles_fixed.csv...

Przykładowe dane po naprawie
                                               title  year
0  Sons, apprentices and successors in late medie...  2009
1  ‘The Military Mirror of Kai’: Swordsmanship an...  2014
2                            Hedonism Before Bentham  2015

Fragment abstraktu (bez tagów):
Book synopsis: The existence and changing of generations in family life, business and politics was a central feature of towns as well as rural societies in earlier times. Even so, it remains understud...


C:\Users\Greg Z\AppData\Local\Temp\ipykernel_23488\3938103702.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace('nan', np.nan) ### usuwanie stringa "nan"
